# XGBoost — Batch Pipelines (unconstrained baseline)

Same structure as the FairPFN notebook: a small **TRIAL** cell (1-2 files) before each
full batch run. Delete the TRIAL cells once confirmed working — they write to separate
`_TRIAL.csv` files so they never touch your real results.

I've kept the output naming as `XGBoost_syn_results.csv` / `XGBoost_semi_syn_results.csv`
to match the `FairPFN_syn_results` / `FairPFN_semi_syn_results` convention you gave me —
let me know if you'd rather keep the original `xgb_syn_results` / `xgb_semi_syn_results` names.

**Paths:**
- Synthetic data: `Data_generation/SF_and_Hidden_Nodes_Data_Simulation/results/generated_160_csv_10_seeds/full_data`
- Semi-synthetic data: `Data_generation/HR_Simulation_Datasets`
- Results output: `model_results/`


In [ ]:
# ==========================================
# LIBRARIES
# ==========================================
import os
import glob
import re
import sys
import types
import warnings

import pandas as pd
import numpy as np

# --- XGBoost compatibility fix for missing pkg_resources ---
# Jupyter sometimes loads a broken/partial pkg_resources in the background.
# This forces the patch regardless of whether it was already loaded.
if 'pkg_resources' not in sys.modules:
    sys.modules['pkg_resources'] = types.ModuleType('pkg_resources')

import pkg_resources

if not hasattr(pkg_resources, 'DistributionNotFound'):
    class DistributionNotFound(Exception): pass
    pkg_resources.DistributionNotFound = DistributionNotFound

if not hasattr(pkg_resources, 'get_distribution'):
    def get_distribution(name):
        raise pkg_resources.DistributionNotFound()
    pkg_resources.get_distribution = get_distribution
# ----------------------------------------------------------

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

# Shared paths
SYN_INPUT_FOLDER = "Data_generation/SF_and_Hidden_Nodes_Data_Simulation/results/generated_160_csv_10_seeds/full_data"
SEMI_INPUT_FOLDER = "Data_generation/HR_Simulation_Datasets"
OUTPUT_DIR = "model_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## Purely Synthetic Data

### 🧪 TRIAL — run on 1-2 files only (delete this cell once confirmed working)

In [ ]:
# --- TRIAL: synthetic, 2 files only, writes to a separate _TRIAL.csv ---
N_TRIAL_FILES = 2
trial_output_file = os.path.join(OUTPUT_DIR, "XGBoost_syn_results_TRIAL.csv")

dataset_files_trial = glob.glob(os.path.join(SYN_INPUT_FOLDER, "*.csv"))[:N_TRIAL_FILES]
print(f"[TRIAL] Found {len(dataset_files_trial)} file(s) to test with.")

xgb_trial = XGBClassifier(eval_metric='logloss', random_state=42)
trial_results = []

for file_path in dataset_files_trial:
    dataset_name = os.path.basename(file_path)
    print(f"\n[TRIAL] Processing: {dataset_name} ...")

    try:
        df = pd.read_csv(file_path)

        s_cols = [col for col in df.columns if col.startswith('S')]
        x_cols = [col for col in df.columns if col.startswith('X')]
        u_cols = [col for col in df.columns if col.startswith('U') or col.lower() == 'u']

        if 'Y' not in df.columns or len(s_cols) == 0:
            print("  [SKIPPED] Missing 'Y' or 'S' columns.")
            continue

        y_data = df['Y'].values
        s_target = 'S0' if 'S0' in s_cols else s_cols[0]
        A_train = df[[s_target]]
        X_features = df[x_cols]
        X_combined = pd.concat([A_train, X_features], axis=1).values

        train_size = min(1000, int(len(X_combined) * 0.8))
        X_train, X_test, y_train, y_test = train_test_split(
            X_combined, y_data, train_size=train_size, random_state=42, stratify=y_data
        )

        xgb_trial.fit(X_train, y_train)
        probs = xgb_trial.predict_proba(X_test)
        predictions = xgb_trial.predict(X_test)
        prob_preds = probs[:, 1] if len(probs.shape) > 1 else probs

        auc = roc_auc_score(y_test, prob_preds)
        acc = accuracy_score(y_test, predictions)
        prec = precision_score(y_test, predictions, zero_division=0)
        rec = recall_score(y_test, predictions, zero_division=0)
        f1 = f1_score(y_test, predictions, zero_division=0)

        group_1_mask = (X_test[:, 0] == 1)
        group_0_mask = (X_test[:, 0] == 0)
        rate_1 = np.mean(predictions[group_1_mask]) if np.sum(group_1_mask) > 0 else 0
        rate_0 = np.mean(predictions[group_0_mask]) if np.sum(group_0_mask) > 0 else 0
        stat_parity_diff = abs(rate_1 - rate_0)
        disp_impact = (rate_1 / rate_0) if rate_0 > 0 else float('inf')

        y_true_1 = y_test[group_1_mask]
        y_pred_1 = predictions[group_1_mask]
        tpr_1 = recall_score(y_true_1, y_pred_1, zero_division=0) if len(y_true_1) > 0 else 0
        y_true_0 = y_test[group_0_mask]
        y_pred_0 = predictions[group_0_mask]
        tpr_0 = recall_score(y_true_0, y_pred_0, zero_division=0) if len(y_true_0) > 0 else 0
        equal_opp_diff = abs(tpr_1 - tpr_0)

        trial_results.append({
            "model_name": "XGBoost", "name_dataset": dataset_name,
            "n_S": len(s_cols), "n_X": len(x_cols), "n_U": len(u_cols),
            "total_samples": len(df),
            "ROC_AUC": round(auc, 4), "Accuracy": round(acc, 4),
            "Precision": round(prec, 4), "Recall": round(rec, 4), "F1_Score": round(f1, 4),
            "Statistical_Parity_Diff_(ATE)": round(stat_parity_diff, 4),
            "Disparate_Impact_Ratio": round(disp_impact, 4),
            "Equal_Opportunity_Diff": round(equal_opp_diff, 4),
            "Pos_Rate_S1": round(rate_1, 4), "Pos_Rate_S0": round(rate_0, 4)
        })
        print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: {stat_parity_diff:.3f}")

    except Exception as e:
        print(f"  [ERROR] {str(e)}")

trial_df = pd.DataFrame(trial_results)
trial_df.to_csv(trial_output_file, index=False)
print(f"\n[TRIAL] Done. Results written to {trial_output_file}")
trial_df


### Full batch pipeline — Purely Synthetic Data

In [ ]:
# ==========================================
# 1. SETUP & INITIALIZATION
# ==========================================
input_folder = SYN_INPUT_FOLDER
output_file = os.path.join(OUTPUT_DIR, "XGBoost_syn_results.csv")

# Find all CSV datasets in the input folder (ignoring zip files automatically)
dataset_files = glob.glob(os.path.join(input_folder, "*.csv"))
print(f"Found {len(dataset_files)} CSV datasets to process.")

# Initialize the XGBoost model ONCE
print("Initializing XGBoost model...")
xgb_model = XGBClassifier(eval_metric='logloss', random_state=42)

# This list will hold a dictionary of results for each dataset
master_results = []

# ==========================================
# 2. BATCH PROCESSING LOOP
# ==========================================
for file_path in dataset_files:
    dataset_name = os.path.basename(file_path)
    print(f"\nProcessing: {dataset_name} ...")

    try:
        # Load dataset
        df = pd.read_csv(file_path)

        # --- DYNAMIC FEATURE DISCOVERY ---
        s_cols = [col for col in df.columns if col.startswith('S')]
        x_cols = [col for col in df.columns if col.startswith('X')]
        u_cols = [col for col in df.columns if col.startswith('U') or col.lower() == 'u']

        if 'Y' not in df.columns or len(s_cols) == 0:
            print(f"  [SKIPPED] Missing 'Y' or 'S' columns.")
            continue

        # Get target variable
        y_data = df['Y'].values

        # Select Protected Attribute (Enforce S0 if it exists, otherwise grab the first S column)
        s_target = 'S0' if 'S0' in s_cols else s_cols[0]
        A_train = df[[s_target]]

        # Select Observable Features
        X_features = df[x_cols]

        # Combine: S_target MUST be at index 0. All other S and U variables are dropped.
        X_combined = pd.concat([A_train, X_features], axis=1).values

        # --- ML BEST PRACTICES ---
        train_size = min(1000, int(len(X_combined) * 0.8))

        X_train, X_test, y_train, y_test = train_test_split(
            X_combined, y_data, train_size=train_size, random_state=42, stratify=y_data
        )

        # --- MODEL INFERENCE ---
        # Feed the context data so XGBoost can learn the rules
        xgb_model.fit(X_train, y_train)

        # XGBoost handles predictions fast, no chunking needed
        probs = xgb_model.predict_proba(X_test)
        predictions = xgb_model.predict(X_test)
        prob_preds = probs[:, 1] if len(probs.shape) > 1 else probs

        # --- 1. PREDICTION METRICS (Evaluated on the unseen TEST set) ---
        auc = roc_auc_score(y_test, prob_preds)
        acc = accuracy_score(y_test, predictions)
        prec = precision_score(y_test, predictions, zero_division=0)
        rec = recall_score(y_test, predictions, zero_division=0)
        f1 = f1_score(y_test, predictions, zero_division=0)

        # --- 2. FAIRNESS METRICS ---
        # Identify groups based on the protected attribute (Index 0 of TEST set)
        group_1_mask = (X_test[:, 0] == 1)
        group_0_mask = (X_test[:, 0] == 0)

        # Positive prediction rates per group
        rate_1 = np.mean(predictions[group_1_mask]) if np.sum(group_1_mask) > 0 else 0
        rate_0 = np.mean(predictions[group_0_mask]) if np.sum(group_0_mask) > 0 else 0

        # Statistical Parity Difference (ATE) -> Ideal value is 0.0
        stat_parity_diff = abs(rate_1 - rate_0)

        # Disparate Impact Ratio -> Ideal value is 1.0
        disp_impact = (rate_1 / rate_0) if rate_0 > 0 else float('inf')

        # Equal Opportunity Difference (Difference in True Positive Rates) -> Ideal is 0.0
        y_true_1 = y_test[group_1_mask]
        y_pred_1 = predictions[group_1_mask]
        tpr_1 = recall_score(y_true_1, y_pred_1, zero_division=0) if len(y_true_1) > 0 else 0

        y_true_0 = y_test[group_0_mask]
        y_pred_0 = predictions[group_0_mask]
        tpr_0 = recall_score(y_true_0, y_pred_0, zero_division=0) if len(y_true_0) > 0 else 0

        equal_opp_diff = abs(tpr_1 - tpr_0)

        # --- RECORD DATA ---
        master_results.append({
            "model_name": "XGBoost",
            "name_dataset": dataset_name,
            "n_S": len(s_cols),
            "n_X": len(x_cols),
            "n_U": len(u_cols),
            "total_samples": len(df),
            # Prediction Indicators
            "ROC_AUC": round(auc, 4),
            "Accuracy": round(acc, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1_Score": round(f1, 4),
            # Fairness Indicators
            "Statistical_Parity_Diff_(ATE)": round(stat_parity_diff, 4),
            "Disparate_Impact_Ratio": round(disp_impact, 4),
            "Equal_Opportunity_Diff": round(equal_opp_diff, 4),
            "Pos_Rate_S1": round(rate_1, 4),
            "Pos_Rate_S0": round(rate_0, 4)
        })

        print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: {stat_parity_diff:.3f}")

    except Exception as e:
        print(f"  [ERROR] Failed to process. Reason: {str(e)}")

# ==========================================
# 3. SAVE AGGREGATED RESULTS TO CSV
# ==========================================
if len(master_results) > 0:
    results_df = pd.DataFrame(master_results)

    # Check if the file already exists
    file_exists = os.path.isfile(output_file)

    # Save to CSV. Mode 'a' appends if exists, 'w' writes new.
    results_df.to_csv(output_file, mode='a', header=not file_exists, index=False)

    print("\n" + "="*50)
    print(f"ALL JOBS COMPLETE! Processed {len(master_results)} datasets successfully.")
    print(f"Results appended to: {output_file}")
    print("="*50)
else:
    print("\nNo datasets were successfully processed.")


## Semi-Synthetic (HR) Data

### 🧪 TRIAL — run on 1-2 files only (delete this cell once confirmed working)

In [ ]:
# --- TRIAL: semi-synthetic, 2 files only, writes to a separate _TRIAL.csv ---
N_TRIAL_FILES = 2
trial_output_file_semi = os.path.join(OUTPUT_DIR, "XGBoost_semi_syn_results_TRIAL.csv")

dataset_files_trial_semi = glob.glob(os.path.join(SEMI_INPUT_FOLDER, "*.csv"))[:N_TRIAL_FILES]
print(f"[TRIAL] Found {len(dataset_files_trial_semi)} file(s) to test with.")

xgb_trial_semi = XGBClassifier(eval_metric='logloss', random_state=42)
trial_results_semi = []

for file_path in dataset_files_trial_semi:
    dataset_name = os.path.basename(file_path)
    print(f"\n[TRIAL] Processing: {dataset_name} ...")

    try:
        df = pd.read_csv(file_path)

        name_no_ext = dataset_name.replace('.csv', '')
        bias_level = "HighBias" if "HighBias" in name_no_ext else ("LowBias" if "LowBias" in name_no_ext else "Unknown")
        thresh_match = re.search(r'Thresh([\d\.]+)', name_no_ext)
        threshold = thresh_match.group(1) if thresh_match else "Unknown"
        data_type = "BIASED_ALL" if "BIASED_ALL" in name_no_ext else ("FAIR_ALL" if "FAIR_ALL" in name_no_ext else "Unknown")

        y_cols = [col for col in df.columns if col.startswith('Y')]
        if not y_cols:
            print("  [SKIPPED] Missing 'Y' target column.")
            continue
        y_data = df[y_cols[0]].values

        s_target_list = [col for col in df.columns if col.startswith('S')]
        if not s_target_list:
            print("  [SKIPPED] Missing 'S' column.")
            continue
        s_target = s_target_list[0]
        A_train = df[[s_target]]

        x_cols = [col for col in df.columns if col.startswith('X_')]
        X_features = df[x_cols]
        u_cols = [col for col in df.columns if col.startswith(('U_', 'H_', 'C_'))]

        X_combined = pd.concat([A_train, X_features], axis=1).values

        X_train, X_test, y_train, y_test = train_test_split(
            X_combined, y_data, train_size=min(1000, int(len(X_combined) * 0.8)),
            random_state=42, stratify=y_data
        )

        xgb_trial_semi.fit(X_train, y_train)
        probs = xgb_trial_semi.predict_proba(X_test)
        predictions = xgb_trial_semi.predict(X_test)
        prob_preds = probs[:, 1] if len(probs.shape) > 1 else probs

        auc = roc_auc_score(y_test, prob_preds)
        acc = accuracy_score(y_test, predictions)
        prec = precision_score(y_test, predictions, zero_division=0)
        rec = recall_score(y_test, predictions, zero_division=0)
        f1 = f1_score(y_test, predictions, zero_division=0)

        group_1_mask = (X_test[:, 0] == 1)
        group_0_mask = (X_test[:, 0] == 0)
        rate_1 = np.mean(predictions[group_1_mask]) if np.sum(group_1_mask) > 0 else 0
        rate_0 = np.mean(predictions[group_0_mask]) if np.sum(group_0_mask) > 0 else 0
        stat_parity_diff = abs(rate_1 - rate_0)
        disp_impact = (rate_1 / rate_0) if rate_0 > 0 else float('inf')

        tpr_1 = recall_score(y_test[group_1_mask], predictions[group_1_mask], zero_division=0) if np.sum(group_1_mask) > 0 else 0
        tpr_0 = recall_score(y_test[group_0_mask], predictions[group_0_mask], zero_division=0) if np.sum(group_0_mask) > 0 else 0
        equal_opp_diff = abs(tpr_1 - tpr_0)

        trial_results_semi.append({
            "model_name": "XGBoost", "name_dataset": dataset_name,
            "bias_level": bias_level, "threshold": threshold, "data_type": data_type,
            "n_S": len(s_target_list), "n_X": len(x_cols), "n_U": len(u_cols),
            "total_samples": len(df),
            "ROC_AUC": round(auc, 4), "Accuracy": round(acc, 4),
            "Precision": round(prec, 4), "Recall": round(rec, 4), "F1_Score": round(f1, 4),
            "Statistical_Parity_Diff_(ATE)": round(stat_parity_diff, 4),
            "Disparate_Impact_Ratio": round(disp_impact, 4),
            "Equal_Opportunity_Diff": round(equal_opp_diff, 4),
            "Pos_Rate_S1": round(rate_1, 4), "Pos_Rate_S0": round(rate_0, 4)
        })
        print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: {stat_parity_diff:.3f}")

    except Exception as e:
        print(f"  [ERROR] {str(e)}")

trial_df_semi = pd.DataFrame(trial_results_semi)
trial_df_semi.to_csv(trial_output_file_semi, index=False)
print(f"\n[TRIAL] Done. Results written to {trial_output_file_semi}")
trial_df_semi


### Full batch pipeline — Semi-Synthetic (HR) Data

In [ ]:
# ==========================================
# 1. SETUP & INITIALIZATION
# ==========================================
input_folder = SEMI_INPUT_FOLDER
output_file = os.path.join(OUTPUT_DIR, "XGBoost_semi_syn_results.csv")

dataset_files = glob.glob(os.path.join(input_folder, "*.csv"))
print(f"Found {len(dataset_files)} semi-synthetic datasets to process.")

print("Initializing XGBoost model...")
xgb_model = XGBClassifier(eval_metric='logloss', random_state=42)

master_results = []

# ==========================================
# 2. BATCH PROCESSING LOOP
# ==========================================
for file_path in dataset_files:
    dataset_name = os.path.basename(file_path)
    print(f"\nProcessing: {dataset_name} ...")

    try:
        df = pd.read_csv(file_path)

        # --- FILENAME PARAMETER EXTRACTION ---
        name_no_ext = dataset_name.replace('.csv', '')
        bias_level = "HighBias" if "HighBias" in name_no_ext else ("LowBias" if "LowBias" in name_no_ext else "Unknown")
        thresh_match = re.search(r'Thresh([\d\.]+)', name_no_ext)
        threshold = thresh_match.group(1) if thresh_match else "Unknown"
        data_type = "BIASED_ALL" if "BIASED_ALL" in name_no_ext else ("FAIR_ALL" if "FAIR_ALL" in name_no_ext else "Unknown")

        # --- DYNAMIC FEATURE DISCOVERY ---
        y_cols = [col for col in df.columns if col.startswith('Y')]
        if not y_cols:
            print("  [SKIPPED] Missing 'Y' target column.")
            continue

        y_data = df[y_cols[0]].values

        # Identify protected attribute (Look for S or G)
        s_target_list = [col for col in df.columns if col.startswith('S')]
        if not s_target_list:
            print("  [SKIPPED] Missing 'S' column.")
            continue

        s_target = s_target_list[0]
        A_train = df[[s_target]]

        x_cols = [col for col in df.columns if col.startswith('X_')]
        X_features = df[x_cols]
        u_cols = [col for col in df.columns if col.startswith(('U_', 'H_', 'C_'))]

        X_combined = pd.concat([A_train, X_features], axis=1).values

        # --- MODEL INFERENCE ---
        X_train, X_test, y_train, y_test = train_test_split(
            X_combined, y_data, train_size=min(1000, int(len(X_combined) * 0.8)),
            random_state=42, stratify=y_data
        )

        xgb_model.fit(X_train, y_train)

        probs = xgb_model.predict_proba(X_test)
        predictions = xgb_model.predict(X_test)
        prob_preds = probs[:, 1] if len(probs.shape) > 1 else probs

        # --- METRICS ---
        auc = roc_auc_score(y_test, prob_preds)
        acc = accuracy_score(y_test, predictions)
        prec = precision_score(y_test, predictions, zero_division=0)
        rec = recall_score(y_test, predictions, zero_division=0)
        f1 = f1_score(y_test, predictions, zero_division=0)

        # Fairness
        group_1_mask = (X_test[:, 0] == 1)
        group_0_mask = (X_test[:, 0] == 0)
        rate_1 = np.mean(predictions[group_1_mask]) if np.sum(group_1_mask) > 0 else 0
        rate_0 = np.mean(predictions[group_0_mask]) if np.sum(group_0_mask) > 0 else 0

        stat_parity_diff = abs(rate_1 - rate_0)
        disp_impact = (rate_1 / rate_0) if rate_0 > 0 else float('inf')

        tpr_1 = recall_score(y_test[group_1_mask], predictions[group_1_mask], zero_division=0) if np.sum(group_1_mask) > 0 else 0
        tpr_0 = recall_score(y_test[group_0_mask], predictions[group_0_mask], zero_division=0) if np.sum(group_0_mask) > 0 else 0
        equal_opp_diff = abs(tpr_1 - tpr_0)

        master_results.append({
            "model_name": "XGBoost",
            "name_dataset": dataset_name,
            "bias_level": bias_level,
            "threshold": threshold,
            "data_type": data_type,
            "n_S": len(s_target_list),
            "n_X": len(x_cols),
            "n_U": len(u_cols),
            "total_samples": len(df),
            "ROC_AUC": round(auc, 4),
            "Accuracy": round(acc, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1_Score": round(f1, 4),
            "Statistical_Parity_Diff_(ATE)": round(stat_parity_diff, 4),
            "Disparate_Impact_Ratio": round(disp_impact, 4),
            "Equal_Opportunity_Diff": round(equal_opp_diff, 4),
            "Pos_Rate_S1": round(rate_1, 4),
            "Pos_Rate_S0": round(rate_0, 4)
        })

        print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: {stat_parity_diff:.3f}")

    except Exception as e:
        print(f"  [ERROR] Failed to process. Reason: {str(e)}")

# ==========================================
# 3. SAVE AGGREGATED RESULTS TO CSV
# ==========================================
if len(master_results) > 0:
    results_df = pd.DataFrame(master_results)
    file_exists = os.path.isfile(output_file)
    results_df.to_csv(output_file, mode='a', header=not file_exists, index=False)
    print(f"\nALL JOBS COMPLETE! Processed {len(master_results)} datasets. Results saved to {output_file}.")
